## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-25-19-54-52 +0000,bbc,US Army says controlled blast of explosives in...,https://www.bbc.co.uk/news/articles/cwg7l5ve25...
1,2026-11-25-19-54-33 +0000,nypost,Iran to execute 2 protesters at dawn after tor...,https://nypost.com/2026/11/25/world-news/iran-...
2,2026-11-25-19-52-53 +0000,nyt,Sept. 11 Defendants Get New Opening to Challen...,https://www.nytimes.com/2026/11/25/us/politics...
3,2026-11-25-19-50-40 +0000,cbc,"Top Gear is back in 2028, with chef Gordon Ram...",https://www.cbc.ca/news/entertainment/top-gear...
4,2026-11-25-19-44-54 +0000,nypost,Kamala Harris’ much-older ex-lover deals crush...,https://nypost.com/2026/11/25/us-news/ex-sf-ma...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2723/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
61,trump,42
441,nvidia,31
606,china,22
17,new,19
218,huang,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
329,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...,150
204,2026-11-25-11-58-00 +0000,wsj,Nvidia’s Jensen Huang is set for at least two ...,https://www.wsj.com/politics/policy/trump-says...,143
134,2026-11-25-15-46-00 +0000,wsj,There was no evidence a Chinese tech executive...,https://www.wsj.com/politics/national-security...,137
248,2026-11-25-10-00-00 +0000,wapo,"In backing Nvidia’s Huang, Trump deals his Chi...",https://www.washingtonpost.com/politics/2026/1...,121
88,2026-11-25-17-24-25 +0000,wapo,"Trump proposes new tariffs on China, Europe an...",https://www.washingtonpost.com/politics/2026/1...,119


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
329,150,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...
16,99,2026-11-25-19-25-40 +0000,nypost,NATO boss says Ukraine ‘doing exceptionally we...,https://nypost.com/2026/11/25/us-news/nato-bos...
142,71,2026-11-25-15-25-37 +0000,startribune,‘No risk’ to Minnesota as Army blows up bomb-m...,https://www.startribune.com/iowa-lead-detonati...
253,66,2026-11-25-09-56-46 +0000,wsj,Stock Market Today: Tech-Led Slide Deepens as ...,https://www.wsj.com/livecoverage/stock-market-...
145,62,2026-11-25-15-10-37 +0000,wapo,Kilmar Abrego García arrives in Spain to start...,https://www.washingtonpost.com/immigration/202...
306,46,2026-11-25-01-55-34 +0000,latimes,Thomas Markle indicted on child porn charges; ...,https://www.latimes.com/entertainment-arts/sto...
291,44,2026-11-25-03-00-00 +0000,wsj,A Democratic push to enact Medicare for All is...,https://www.wsj.com/politics/policy/democrats-...
330,43,2026-11-25-00-13-52 +0000,nypost,"Andrew Giuliani’s daughter, 4, steals spotligh...",https://nypost.com/2026/11/24/us-news/andrew-g...
15,42,2026-11-25-19-26-44 +0000,bbc,Two children among six dead after plane crash ...,https://www.bbc.co.uk/news/articles/c6z74l1017...
436,41,2026-11-24-20-08-46 +0000,latimes,High tide breaks 44-year-old record in S.F. as...,https://www.latimes.com/california/story/2026-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
